In [2]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_excel("supply_chain_inventory_data.xlsx")

# Show output
df.head()


,Item_ID,Item_Name,Daily_Demand,Current_Stock,Reorder_Level,Reorder_Quantity,Lead_Time_Days,Supplier,Warehouse,Last_Updated
0,ITEM-0001,Finished Good 2,181,2014,2353,3090,13,Supplier Alpha,Warehouse East,2026-01-12
1,ITEM-0002,Finished Good 1,200,1563,2000,3424,10,Supplier Gamma,Warehouse South,2026-01-12
2,ITEM-0003,Finished Good 1,20,267,220,420,11,Supplier Alpha,Warehouse East,2026-01-12
3,ITEM-0004,Finished Good 1,74,398,518,570,7,Supplier Gamma,Warehouse East,2026-01-12
4,ITEM-0005,Finished Good 2,42,127,294,429,7,Supplier Beta,Warehouse East,2026-01-12


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Item_ID           1000 non-null   object
 1   Item_Name         1000 non-null   object
 2   Daily_Demand      1000 non-null   int64 
 3   Current_Stock     1000 non-null   int64 
 4   Reorder_Level     1000 non-null   int64 
 5   Reorder_Quantity  1000 non-null   int64 
 6   Lead_Time_Days    1000 non-null   int64 
 7   Supplier          1000 non-null   object
 8   Warehouse         1000 non-null   object
 9   Last_Updated      1000 non-null   object
dtypes: int64(5), object(5)
memory usage: 78.3+ KB


In [6]:
# Missing values report
df.isna().sum().sort_values(ascending=False)

Item_ID             0
Item_Name           0
Daily_Demand        0
Current_Stock       0
Reorder_Level       0
Reorder_Quantity    0
Lead_Time_Days      0
Supplier            0
Warehouse           0
Last_Updated        0
dtype: int64

In [8]:
# Check duplicates
df.duplicated().sum()

np.int64(0)

In [16]:
df['Last_Updated'] = pd.to_datetime(df['Last_Updated'])

In [20]:
df['Item_Name'] = df['Item_Name'].str.strip().str.title()


In [22]:
df = df.drop_duplicates()


In [24]:
df.rename(columns={
    'Daily_Demand': 'Demand',
    'Current_Stock': 'Stock'
}, inplace=True)

In [26]:
df.isna().sum()

Item_ID             0
Item_Name           0
Demand              0
Stock               0
Reorder_Level       0
Reorder_Quantity    0
Lead_Time_Days      0
Supplier            0
Warehouse           0
Last_Updated        0
dtype: int64

In [28]:
df['Low_Stock_Flag'] = np.where(df['Stock'] < df['Reorder_Level'], 1, 0)

In [30]:
df['Stock_Coverage_Days'] = df['Stock'] / df['Demand']

In [32]:
df['Demand_Group'] = pd.qcut(df['Demand'], q=3, labels=['Low', 'Medium', 'High'])

In [34]:
df.groupby('Warehouse')['Stock'].mean().sort_values(ascending=False)


Warehouse
Warehouse East     1018.392857
Warehouse South    1014.573333
Warehouse North     946.304945
Name: Stock, dtype: float64

In [36]:
df['Supplier'].value_counts()

Supplier
Supplier Alpha    337
Supplier Beta     333
Supplier Gamma    330
Name: count, dtype: int64

In [38]:
df['Low_Stock_Flag'].value_counts(normalize=True) * 100

Low_Stock_Flag
0    52.1
1    47.9
Name: proportion, dtype: float64

In [43]:

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error


In [45]:
X = df[['Demand', 'Reorder_Level', 'Lead_Time_Days', 'Supplier', 'Warehouse']]
y = df['Stock']


In [47]:
cat_cols = ['Supplier', 'Warehouse']
num_cols = ['Demand', 'Reorder_Level', 'Lead_Time_Days']

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
    ('num', 'passthrough', num_cols)
])


In [49]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [51]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

model.fit(X_train, y_train)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Supplier', 'Warehouse']),
                                                 ('num', 'passthrough',
                                                  ['Demand', 'Reorder_Level',
                                                   'Lead_Time_Days'])])),
                ('regressor', LinearRegression())])

In [53]:
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
mae


498.40753750649145

In [55]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, precision_score, recall_score

In [57]:
X = df[['Demand', 'Reorder_Level', 'Lead_Time_Days', 'Supplier', 'Warehouse']]
y = df['Low_Stock_Flag']

In [59]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [61]:
clf = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

clf.fit(X_train, y_train)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Supplier', 'Warehouse']),
                                                 ('num', 'passthrough',
                                                  ['Demand', 'Reorder_Level',
                                                   'Lead_Time_Days'])])),
                ('classifier', LogisticRegression(max_iter=1000))])

In [65]:
y_pred = clf.predict(X_test)

In [67]:
confusion_matrix(y_test, y_pred)

array([[61, 34],
       [72, 33]])

In [69]:
precision_score(y_test, y_pred), recall_score(y_test, y_pred)

(0.4925373134328358, 0.3142857142857143)